1. Import

In [5]:
import os
os.environ["ACCELERATE_DISABLE_RICH"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Đăng nhập Hugging Face bằng Token đã lưu trong Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Cấu hình model
MODEL_NAME = "google/gemma-2-2b-it"
OUTPUT_DIR = "gemma2-movie-chatbot"

2. Dataset

In [6]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            print(os.path.join(dirname, filename))

# Đọc dữ liệu
df = pd.read_csv('/kaggle/input/datasets/hoaquyen666/gemma-finetune-movie-mood/gemma_finetune_movie_mood_dataset_v3_with_edge_cases.csv')

# Làm sạch tên cột (nếu có lỗi khoảng trắng hoặc ký tự BOM-Byte Order Mark)
df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]

# Chuyển DataFrame thành Dataset của Hugging Face
dataset = Dataset.from_pandas(df)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Hàm định dạng dữ liệu theo chuẩn Chat Template của Gemma
def format_chat_template(row):
    messages = [
        {"role": "user", "content": str(row["user"])},
        {"role": "assistant", "content": str(row["assistant"])}
    ]
    # Gemma dùng <start_of_turn> và <end_of_turn>
    # apply_chat_template sẽ tự động thêm các token này
    row["text"] = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return row

dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)

print("Ví dụ dữ liệu sau khi format:")
print(dataset[0]['text'])

/kaggle/input/datasets/hoaquyen666/gemma-finetune-movie-mood/gemma_finetune_movie_mood_dataset_v3_with_edge_cases.csv


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Map:   0%|          | 0/6400 [00:00<?, ? examples/s]

Ví dụ dữ liệu sau khi format:
<bos><start_of_turn>user
Đang bực mình quá nên muốn xem phim trả thù cho hả dạ<end_of_turn>
<start_of_turn>model
Trông bạn có vẻ đang bốc hỏa! Xem ngay những bộ phim cực 'cháy' này để xả hết bực dọc nhé. <recommend>mood='tức giận, xả stress', keywords='hài hước đen, trả thù, hài nhảm, phá hoại'</recommend><end_of_turn>



3. Load Model và cấu hình QLoRA

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
import torch

# Cấu hình lượng tử hóa 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load Model
# Đổi device_map="auto" thành device_map={"": 0} để ép model chạy trên 1 GPU duy nhất
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Tắt cache để tương thích với gradient checkpointing/k-bit training
model.config.use_cache = False

# Chuẩn bị model cho Kbit training
model = prepare_model_for_kbit_training(model)

# Cấu hình LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

4. Cấu hình + fine-tune

In [8]:
from trl import SFTTrainer, SFTConfig

# Sử dụng SFTConfig của trl thay vì TrainingArguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,      
    gradient_accumulation_steps=8,      
    optim="paged_adamw_32bit",
    save_steps=100,                     
    logging_steps=10,                   
    learning_rate=1e-5,
    max_steps=300,                      
    bf16=True,                          
    max_grad_norm=0.3,
    warmup_steps=10,                     
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none"
    # Đã xóa max_seq_length
)

# Khởi tạo Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# Bắt đầu huấn luyện
print("Bắt đầu Fine-tuning...")
trainer.train()

# Lưu model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Huấn luyện hoàn tất và đã lưu model!")

Adding EOS to train dataset:   0%|          | 0/6400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6400 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Bắt đầu Fine-tuning...


Step,Training Loss
10,5.501184
20,4.873646
30,4.283608
40,3.795456
50,3.366674
60,2.995246
70,2.781888
80,2.517871
90,2.378497
100,2.281029


Huấn luyện hoàn tất và đã lưu model!


5. Test thử Chatbot sau khi Fine-tune

In [9]:
import re
import torch

def get_movie_recommendation(user_mood):
    model.eval()

    # Tạo prompt đúng chat template Gemma
    messages = [{"role": "user", "content": user_mood}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.25,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Chỉ decode phần model vừa sinh, không decode lại prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    assistant_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return assistant_response

# Test
test_input = "Tôi buồn quá, gợi ý phim nào vui vẻ đi"
result = get_movie_recommendation(test_input)

print(f"User: {test_input}")
print(f"Chatbot: {result}")

# Demo trích xuất thông tin để truy vấn Database
print("\n--- TRÍCH XUẤT DATABASE ---")
match = re.search(r"<recommend>(.*?)</recommend>", result)
if match:
    recommend_tag = match.group(1)

    mood_match = re.search(r"mood='(.*?)'", recommend_tag)
    keywords_match = re.search(r"keywords='(.*?)'", recommend_tag)

    db_mood = mood_match.group(1) if mood_match else None
    db_keywords = keywords_match.group(1) if keywords_match else None

    print(f"Cảm xúc phân loại (Mood): {db_mood}")
    print(f"Từ khóa truy vấn (Keywords): {db_keywords}")
    print("=> Sẽ đưa 2 biến này vào câu lệnh SQL truy vấn Database gợi ý phim!")
else:
    print("Không tìm thấy thẻ recommend.")


User: Tôi buồn quá, gợi ý phim nào vui vẻ đi
Chatbot: Chào bạn, cảm giác đó rất bình thường. Cùng xem một bộ phim sẽ giúp bạn giải tỏa nhé! <recommend>mood='buồn bã', keywords='thành công, hài hước'</recommend> 

**Những câu chuyện đầy hy vọng:**

* **The Pursuit of Happyness (2006):** Một nhân vật có hoàn cảnh khó khăn nhưng vẫn lạc quan trong cuộc sống. Trưởng thành không chỉ là về tiền bạc mà còn phải tìm kiếm hạnh phúc đích thực.
* **Hidden Figures (2016):** Khát khao lập gia đình và đạt được ước mơ của những người phụ nữ tài năng đã tạo nên lịch sử khoa học vĩ đại. M

--- TRÍCH XUẤT DATABASE ---
Cảm xúc phân loại (Mood): buồn bã
Từ khóa truy vấn (Keywords): thành công, hài hước
=> Sẽ đưa 2 biến này vào câu lệnh SQL truy vấn Database gợi ý phim!


In [14]:
import torch
import shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = MODEL_NAME
ADAPTER_DIR = OUTPUT_DIR
MERGED_DIR = "/kaggle/working/gemma2-movie-chatbot-merged"

# Load lại base model ở dạng bf16/fp16 để merge LoRA vào
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load LoRA adapter đã train
merged_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

# Merge LoRA vào base model
merged_model = merged_model.merge_and_unload()

# Lưu model đã merge
merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="2GB"
)

tokenizer.save_pretrained(MERGED_DIR)

print("Đã merge model vào:", MERGED_DIR)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

Đã merge model vào: /kaggle/working/gemma2-movie-chatbot-merged


In [20]:
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
  /kaggle/working/gemma2-movie-chatbot-merged \
  --outfile /kaggle/working/gemma2-movie-chatbot-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: gemma2-movie-chatbot-merged
INFO:hf-to-gguf:Model architecture: Gemma2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> F16, shape = {2304, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> F16, shape = {9216, 2304}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> F16, shape = {2304,

In [21]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
  /kaggle/working/gemma2-movie-chatbot-f16.gguf \
  /kaggle/working/gemma2-movie-chatbot-Q4_K_M.gguf \
  Q4_K_M

llama_print_build_info: build = 9855 (6dbc1174b)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/gemma2-movie-chatbot-f16.gguf' to '/kaggle/working/gemma2-movie-chatbot-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 288 tensors from /kaggle/working/gemma2-movie-chatbot-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gemma2 Movie Chatbot Merged
llama_model_loader: - kv   3:                         general.size_label str              = 2.6B
llama_model_loader: - kv   4:                      gemma2.context_length u32      